#### Define a few parameters and functions

##### Requirements
This python 3 notebook should only require numpy, pandas, and scipy to do the data analysis

It was originally run with python 3.11.5, numpy 1.24.3, scipy 1.11.3 and pandas 2.1.1

**Note:** *The Corey shape factor and Shape correction factor in this spreadsheet differ slightly from that given in the main text because the Corey shape factor here is calculated from the mean values of A, B and C taken from a sample of 1600 grains in order to illustrate how to calculate the shape correction factor from A, B and C. In the main text, the slightly different Corey shape factor is calculated as the mean value of the Corey shape factors of all the grains in the sample of natural gravel.*


In [ ]:
import numpy as np, pandas as pd

# Configure pandas display options to avoid scientific notation
pd.set_option('display.float_format', '{:.4g}'.format)

# set constants: gravitational acceleration (m/s^2), kinematic viscosity (m^2/s), density of water (kg/m^3)
# and angle of repose for ideal spheres (degrees)
g, nu, rho_f = 9.81, 1e-6, 990.0
angle_of_repose_ideal_spheres = 24

def drag_coeff_calc(sediment_density, VES_diameter, settling_velocity):
    drag_coeff = 4*(sediment_density/rho_f - 1)*g*VES_diameter / (3*settling_velocity**2)
    return drag_coeff

def VES_settling_velocity_calc(sediment_density, VES_diameter):
    # function to calculate settling velocity of sphere as per Dietrich et al,
    def R1(logdx):
        a, b, c, d, f = -3.815642429922084, 1.9459392605305115, -0.09016149260089677, -0.008555578212166149, 0.000757318175522988
        return (a + b*logdx + c*logdx**2 + d*logdx**3 + f*logdx**4)

    dx_veqs = (sediment_density - rho_f) * g * VES_diameter**3 / (rho_f*nu*nu)
    ws_sphere = (10**R1(np.log10(dx_veqs)) * (sediment_density - rho_f)*g*nu/rho_f)**(1/3)
    return ws_sphere

def C_star_calc(sediment_density, grain_volume, settling_velocity, A, B, C):
    VES_diameter = (6*grain_volume/np.pi)**(1/3)  # calculate VES diameter from grain volume
    corey_shape_factor = C/np.sqrt(A*B)
    CD = drag_coeff_calc(sediment_density, VES_diameter, settling_velocity) * corey_shape_factor
    VES_settling_velocity = VES_settling_velocity_calc(sediment_density, VES_diameter)
    Co = drag_coeff_calc(sediment_density, VES_diameter, VES_settling_velocity)
    C_star = CD/Co
    return C_star

def mu_star_calc(angle_of_repose, flume_slope):
    mu_sphere = (np.tan(np.radians(angle_of_repose_ideal_spheres)) - flume_slope)
    mu_star = (np.tan(np.radians(angle_of_repose)) - flume_slope)/mu_sphere
    return mu_star

#### Enter data

An example dataset is given below that is used to demonstrate how to calculate grain shape correction factor $F_{shape}$. The function can be used on your own data if you create a dataframe with the same columns with the same names.

| row name | parameter | units|
|-------------|-----------|------------|
| flume_slope| Flume Slope | m/m |
| sediment_density | Sediment Density| kg/m³ |
| long_grain_axis | Long Grain Axis, A | m|
| middle_grain_axis | Middle Grain Axis, B | m |
| short_grain_axis | Short Grain Axis, C | m|
| grain_volume | Average grain volume | m^3 |
| settling_velocity | Still water settling velocity | m/s |
| angle_of_repose | Angle of repose | degrees |


In [ ]:
data = {
    'spheres': {
        'flume_slope': 0.02199469431199964,
        'sediment_density': 2557.6645329369226, # kg/m3
        'long_grain_axis': 0.0049, # meters
        'middle_grain_axis': 0.0049, # meters
        'short_grain_axis': 0.0049, # meters
        'grain_volume': 6.331367e-8, # m^3
        'settling_velocity': 0.4856, # m/s
        'angle_of_repose': 24.73, # degress
    },
    
    'natural_gravel': {
        'flume_slope': 0.02199469431199964,
        'sediment_density': 2470.921300760533, # kg/m3
        'long_grain_axis': 0.005775, # meters
        'middle_grain_axis': 0.004380, # meters
        'short_grain_axis': 0.003349, # meters
        'grain_volume': 3.635164e-8, # m^3
        'settling_velocity': 0.3133, # m/s
        'angle_of_repose': 35.39, # degrees
    }
}

df = pd.DataFrame(data)
df

#### Calculate grain shape correction factor

In [ ]:
for name, group in df.T.groupby(df.T.index):
    mu_star = mu_star_calc(group.angle_of_repose, group.flume_slope)
    C_star = C_star_calc(group.sediment_density, group.grain_volume, group.settling_velocity, group.long_grain_axis, group.middle_grain_axis, group.short_grain_axis)
    df.loc['corey_shape_factor', name] = (group.short_grain_axis / np.sqrt(group.long_grain_axis * group.middle_grain_axis)).iloc[0]
    df.loc['F_shape', name] = (C_star / mu_star).iloc[0]
    df.loc['C_star', name] = (C_star).iloc[0]
    df.loc['mu_star', name] = (mu_star).iloc[0]

df